In [34]:
import pandas as pd

To read and clean the data individually, with the ultimate goal to make combined survey file

In [35]:
df_2014 = pd.read_csv("../data/processed/extracted/2014.csv")

In [36]:
display(df_2014.head()) # target column - treatment

display(df_2014.dtypes) # looks good

print("NA values observed")
display(df_2014.isna().sum())

display(df_2014.nunique())

display(df_2014.duplicated().sum() ) # zero duplicates

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


Timestamp                      str
Age                          int64
Gender                         str
Country                        str
state                          str
self_employed                  str
family_history                 str
treatment                      str
work_interfere                 str
no_employees                   str
remote_work                    str
tech_company                   str
benefits                       str
care_options                   str
wellness_program               str
seek_help                      str
anonymity                      str
leave                          str
mental_health_consequence      str
phys_health_consequence        str
coworkers                      str
supervisor                     str
mental_health_interview        str
phys_health_interview          str
mental_vs_physical             str
obs_consequence                str
comments                       str
dtype: object

NA values observed


Timestamp                       0
Age                             0
Gender                          0
Country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

Timestamp                    1246
Age                            53
Gender                         49
Country                        48
state                          45
self_employed                   2
family_history                  2
treatment                       2
work_interfere                  4
no_employees                    6
remote_work                     2
tech_company                    2
benefits                        3
care_options                    3
wellness_program                3
seek_help                       3
anonymity                       3
leave                           5
mental_health_consequence       3
phys_health_consequence         3
coworkers                       3
supervisor                      3
mental_health_interview         3
phys_health_interview           3
mental_vs_physical              3
obs_consequence                 2
comments                      160
dtype: int64

np.int64(0)

In [37]:
for col in df_2014.columns:
    if df_2014[col].nunique() <= 60:
        print(f"--- {col}")
        print(df_2014[col].value_counts(dropna=False))
        print()

--- Age
Age
 29             85
 32             82
 26             75
 27             71
 33             70
 28             68
 31             67
 34             65
 30             63
 25             61
 35             55
 23             51
 24             46
 37             43
 38             39
 36             37
 39             33
 40             33
 43             28
 41             21
 22             21
 42             20
 21             16
 46             12
 45             12
 44             11
 19              9
 18              7
 50              6
 48              6
 20              6
 51              5
 56              4
 49              4
 54              3
 55              3
 57              3
 60              2
 47              2
-29              1
 329             1
 99999999999     1
 58              1
 62              1
 65              1
-1726            1
 5               1
 53              1
 61              1
 8               1
 11              1
-1               1


Ages "-29, 99999999999, -1, -1726, 11" - filter all between 14 and 100


Gender: M - "Male" "male" "M" "m" "make" "Cis Male" "Man" "maile" "Mal" "Male (CIS)" "msle" "Mail" "cis male" "Malr" "Cis Man" "ostensibly male, unsure what that really means"

Gender: F - "Female "female" "F" "f" "Woman" "Female" "Cis Female" "Femake" "woman" "cis-female/femme" "Female (cis)"

In question: "Female (trans)", "Trans-female" "Trans woman" - woman (will it inflate the statistics, as people that questioned their gender are usually more affected, should I group them to LGBTQ+/not specified - so in total 4 groups? is this too discriminatory?)

Country : Bahamas, The - the bahamas?

State: NaN for non-US;
work-interfere: Never -> Rarely -> Sometimes -> Often

no_employees - 1-5 -> 6-25 -> 26-100 -> 100-500 -> 500-1000 -> More than 1000

Don't know, not sure, Maybe - same

Leave - Very easy -> Somewhat easy -> Somewhat difficult -> Very difficult

Coworkers - what was the question? 

Supervisor -  ?

obs_consequence -?

Start the filtering:

In [38]:
# Ages, duplicates, drop comments

n_before = len(df_2014)
df_2014 = df_2014[df_2014["Age"].between(14, 100)].copy()
print("Rows removed:", n_before - len(df_2014))       # expect 8

df_2014 = df_2014.drop(columns="comments")
df_2014["Timestamp"] = pd.to_datetime(df_2014["Timestamp"])
print("Duplicate rows:", df_2014.duplicated().sum())
print(df_2014["Age"].describe())

Rows removed: 8
Duplicate rows: 0
count    1251.000000
mean       32.076739
std         7.288272
min        18.000000
25%        27.000000
50%        31.000000
75%        36.000000
max        72.000000
Name: Age, dtype: float64


In [39]:
# genders to Male, Female, Other, Missing

print("Before grouping: ", df_2014["Gender"].value_counts(dropna=False))

male = {"male", "m", "make", "cis male", "man", "maile", "mal", "male (cis)",
        "msle", "mail", "malr", "cis man",
        "ostensibly male, unsure what that really means"}
female = {"female", "f", "woman", "femake", "cis-female/femme", "female (cis)",
          "cis female", "femail"}
uninformative = {"a little about you", "p"}

# treat "nah", "all" as other

def group_gender(g):
    g = g.strip().lower()
    if g in male:            return "male"
    if g in female:          return "female"
    if g in uninformative:   return None          # becomes NaN
    return "other"                       # trans, non-binary, queer, ambiguous

df_2014["Gender"] = df_2014["Gender"].map(group_gender)

print("After grouping: ", df_2014["Gender"].value_counts(dropna=False))



Before grouping:  Gender
Male                                              612
male                                              204
Female                                            121
M                                                 116
female                                             62
F                                                  38
m                                                  34
f                                                  15
Make                                                4
Woman                                               3
Male                                                3
Cis Male                                            2
Female                                              2
Man                                                 2
Female (trans)                                      2
Male-ish                                            1
maile                                               1
Trans-female                                        1
Cis

In [ ]:
# Unsure labels, NA, and ordering

# one label for the "third answer"
df_2014 = df_2014.replace({"Don't know": "Unsure", "Not sure": "Unsure", "Maybe": "Unsure"})

# NA in work_interfere = not applicable (no condition), too many to drop
df_2014["work_interfere"] = df_2014["work_interfere"].fillna("Not applicable")

# keep the orders for later, apply to the ones with a clean order now
ORDER_WORK  = ["Never", "Rarely", "Sometimes", "Often"]
ORDER_LEAVE = ["Very easy", "Somewhat easy", "Somewhat difficult", "Very difficult"]
ORDER_EMP   = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]

df_2014["no_employees"] = pd.Categorical(df_2014["no_employees"], categories=ORDER_EMP, ordered=True)

print(df_2014["no_employees"].value_counts(sort=False))

no_employees
1-5               158
6-25              289
26-100            288
100-500           175
500-1000           60
More than 1000    281
Name: count, dtype: int64


In [54]:
print(df_2014.isna().sum()[lambda s: s > 0])     # what's still missing
# NA for non-US is okay, so do not remove;
# Self-employed -? 
print(df_2014["self_employed"].value_counts(dropna=False))
display(df_2014[df_2014["self_employed"].isna()]) # also looks completely okay for other entries
# Not available, early timestamps (if compared to the data, so probably whis question was added to the survey later)

# most answers are No, so it qouldn't generate a lot of noise to just assume as no, but decided to remove this rows - only 18/~1200

n_before = len(df_2014)
df_2014 = df_2014.dropna(subset=["self_employed"])
print("Rows removed:", n_before - len(df_2014))      # expect 18


state            513
self_employed     18
dtype: int64
self_employed
No     1091
Yes     142
NaN      18
Name: count, dtype: int64


,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence
0,2014-08-27 11:29:31,37,female,United States,IL,NaN,No,Yes,Often,6-25,...,Yes,Somewhat easy,No,No,Some of them,Yes,No,Unsure,Yes,No
1,2014-08-27 11:29:37,44,male,United States,IN,NaN,No,No,Rarely,More than 1000,...,Unsure,Unsure,Unsure,No,No,No,No,No,Unsure,No
2,2014-08-27 11:29:44,32,male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Unsure,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No
3,2014-08-27 11:29:46,31,male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,No,Somewhat difficult,Yes,Yes,Some of them,No,Unsure,Unsure,No,Yes
4,2014-08-27 11:30:22,31,male,United States,TX,NaN,No,No,Never,100-500,...,Unsure,Unsure,No,No,Some of them,Yes,Yes,Yes,Unsure,No
5,2014-08-27 11:31:22,33,male,United States,TN,NaN,Yes,No,Sometimes,6-25,...,Unsure,Unsure,No,No,Yes,Yes,No,Unsure,Unsure,No
6,2014-08-27 11:31:50,35,female,United States,MI,NaN,Yes,Yes,Sometimes,1-5,...,No,Somewhat difficult,Unsure,Unsure,Some of them,No,No,No,Unsure,No
7,2014-08-27 11:32:05,39,male,Canada,NaN,NaN,No,No,Never,1-5,...,Yes,Unsure,No,No,No,No,No,No,No,No
8,2014-08-27 11:32:39,42,female,United States,IL,NaN,Yes,Yes,Sometimes,100-500,...,No,Very difficult,Unsure,No,Yes,Yes,No,Unsure,No,No
9,2014-08-27 11:32:43,23,male,Canada,NaN,NaN,No,No,Never,26-100,...,Unsure,Unsure,No,No,Yes,Yes,Unsure,Unsure,Yes,No


Rows removed: 18


In [57]:
# see the value counts:

for col in df_2014.columns:
    if df_2014[col].nunique() <= 10:
        print(f"--- {col}")
        print(df_2014[col].value_counts(dropna=False))
        print()

--- Gender
Gender
male      973
female    242
other      18
Name: count, dtype: int64

--- self_employed
self_employed
No     1091
Yes     142
Name: count, dtype: int64

--- family_history
family_history
No     751
Yes    482
Name: count, dtype: int64

--- treatment
treatment
Yes    623
No     610
Name: count, dtype: int64

--- work_interfere
work_interfere
Sometimes         457
Not applicable    262
Never             206
Rarely            170
Often             138
Name: count, dtype: int64

--- no_employees
no_employees
6-25              283
26-100            283
More than 1000    280
100-500           172
1-5               156
500-1000           59
Name: count, dtype: int64

--- remote_work
remote_work
No     868
Yes    365
Name: count, dtype: int64

--- tech_company
tech_company
Yes    1009
No      224
Name: count, dtype: int64

--- benefits
benefits
Yes       466
Unsure    400
No        367
Name: count, dtype: int64

--- care_options
care_options
No        492
Yes       434
Unsure 

For "Some of them"" option, the questions below were given

coworkers: Would you be willing to discuss a mental health issue with your coworkers?

supervisor: Would you be willing to discuss a mental health issue with your direct supervisor(s)?

So the answer is plausible.




In [60]:
display(df_2014.head())

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence
18,2014-08-27 11:34:53,46,male,United States,MD,Yes,Yes,No,Sometimes,1-5,...,Yes,Very easy,No,No,Yes,Yes,No,Yes,Yes,Yes
19,2014-08-27 11:35:08,36,male,France,NaN,Yes,Yes,No,Not applicable,6-25,...,Yes,Somewhat easy,No,No,Some of them,Some of them,Unsure,Unsure,Unsure,No
20,2014-08-27 11:35:12,29,male,United States,NY,No,Yes,Yes,Sometimes,100-500,...,No,Somewhat difficult,Unsure,No,Some of them,Some of them,No,No,No,No
21,2014-08-27 11:35:24,31,male,United States,NC,Yes,No,No,Never,1-5,...,Yes,Somewhat difficult,No,No,Some of them,Some of them,No,Unsure,Yes,No
22,2014-08-27 11:35:48,46,male,United States,MA,No,No,Yes,Often,26-100,...,Unsure,Unsure,Unsure,No,Some of them,Yes,No,Unsure,No,No


# Comments about the code:

1. A warning for the modelling stage

State isn’t quite “NaN for non-US”: 11 US respondents have no state, and 3 non-US respondents do. 
Probably we won't use state as a predictor, so that is okay for our data.

Country names (48 distinct, including Bahamas, The) will be looked when the merge will be done (as well as filtering the given answers for Gender - if other categories will arise, it will be easier to add them and not rerruning the whole list)

